# Tarea 2: Segmentación de Clientes
## Notebook 04 — Customer Segmentation using KMeans + PCA

**Objetivo:** Segmentar la base de clientes en grupos homogéneos para orientar la estrategia comercial de easyMoney.

**Metodología:**
- Proceso iterativo de feature engineering en 3 pasos documentados
- KMeans++ con selección de k óptimo (Elbow + Silhouette + Davies-Bouldin + Calinski-Harabasz)
- PCA para visualización 2D orientativa
- Profiling de clusters para interpretación de negocio

**Input:** `master_df_flags.parquet`
**Output:** `customer_segments.csv`, `cluster_profiles.csv`

## 1. Importación de Librerías

In [22]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import os
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
K_FINAL = 7  # k optimo — justificado en Seccion 7.3

print("All imports OK")

All imports OK


## 2. Carga de Datos

In [23]:
df = pd.read_parquet('../../data/processed/master_df_flags.parquet')

# Filtrar registros anómalos — consistente con notebooks 02 y 03
anomaly_flags = ['age_anomaly', 'deceased_anomaly', 'entry_date_anomaly', 'salary_anomaly']
df_clean = df[~df[anomaly_flags].any(axis=1)]

print(f"Registros totales:   {len(df):,}")
print(f"Registros limpios:   {len(df_clean):,}  ({len(df_clean)/len(df)*100:.2f}%)")
print(f"Registros excluidos: {len(df) - len(df_clean):,}  ({(1 - len(df_clean)/len(df))*100:.4f}%)")

# Features temporales: extraidas de la serie completa limpia ANTES de reducir al snapshot
temporal_features = (
    df_clean.sort_values(['pk_cid', 'pk_partition'])
    .groupby('pk_cid')
    .agg(
        active_rate    = ('active_customer', 'mean'),
        active_last3   = ('active_customer', lambda x: x.iloc[-3:].mean() if len(x) >= 3 else x.mean()),
        active_first3  = ('active_customer', lambda x: x.iloc[:3].mean()  if len(x) >= 3 else x.mean()),
        products_first = ('total_products',  'first'),
        products_last  = ('total_products',  'last'),
    )
    .assign(
        activity_trend = lambda d: d['active_last3']  - d['active_first3'],
        product_growth = lambda d: d['products_last'] - d['products_first'],
    )
    .drop(columns=['active_last3', 'active_first3', 'products_first', 'products_last'])
    .reset_index()
)

df_latest = (
    df_clean[df_clean['pk_partition'] == df_clean['pk_partition'].max()]
    .merge(temporal_features, on='pk_cid', how='left')
)

print(f"\nShape (todas las particiones, limpio): {df_clean.shape}")
print(f"Shape (ultimo snapshot, limpio):       {df_latest.shape}")
print(f"Clientes unicos:                       {df_latest['pk_cid'].nunique()}")
print(f"Particion seleccionada:                {df_latest['pk_partition'].unique()}")
print(f"Features temporales anadidas:          active_rate, activity_trend, product_growth")
df_latest.head(3)

Registros totales:   5,962,924
Registros limpios:   5,940,692  (99.63%)
Registros excluidos: 22,232  (0.3728%)

Shape (todas las particiones, limpio): (5940692, 38)
Shape (ultimo snapshot, limpio):       (441752, 41)
Clientes unicos:                       441752
Particion seleccionada:                <DatetimeArray>
['2019-05-28 00:00:00']
Length: 1, dtype: datetime64[ns]
Features temporales anadidas:          active_rate, activity_trend, product_growth


,pk_cid,pk_partition,entry_date,entry_channel,active_customer,segment,short_term_deposit,loans,mortgage,funds,...,client_age_months,age_group,salary_group,age_anomaly,deceased_anomaly,entry_date_anomaly,salary_anomaly,active_rate,activity_trend,product_growth
0,16063,2019-05-28,2018-11-19,KAT,0.0,02 - PARTICULARES,0,0,0,0,...,6,55-65,80-120k,False,False,False,False,0.714286,-0.666667,0
1,16203,2019-05-28,2018-12-23,KAT,1.0,01 - TOP,0,0,0,0,...,5,65+,80-120k,False,False,False,False,0.833333,0.333333,1
2,16502,2019-05-28,2018-09-30,KHN,1.0,02 - PARTICULARES,0,0,0,0,...,8,55-65,80-120k,False,False,False,False,1.000000,0.000000,1


### Nota sobre el universo de clientes

El dataset historico contiene **456,318 clientes unicos** a lo largo de todas las particiones.
El ultimo snapshot (mayo 2019) tiene **442,995 clientes** — una diferencia de **13,323 clientes**
que no aparecen en la ultima particion, indicando probable churn antes del corte de datos.

La segmentacion usa exclusivamente el ultimo snapshot porque:
- Representa el estado actual de cada cliente
- Evita duplicados (un cliente por fila)
- Es el universo accionable para campanas de marketing

## 3. Exploración Inicial del Dataset

In [24]:
print("Columnas disponibles:")
print(df_latest.columns.tolist())

print()
nulos = df_latest.isnull().sum()
print("Nulos por columna:")
print(nulos[nulos > 0] if nulos.sum() > 0 else "Sin nulos en el dataset")

print()
print("Valores unicos de variables categoricas clave:")
for col in ['gender', 'segment', 'age_group', 'salary_group']:
    vals = sorted(df_latest[col].dropna().unique())
    print(f"  {col}: {vals}")

Columnas disponibles:
['pk_cid', 'pk_partition', 'entry_date', 'entry_channel', 'active_customer', 'segment', 'short_term_deposit', 'loans', 'mortgage', 'funds', 'securities', 'long_term_deposit', 'credit_card', 'payroll', 'pension_plan', 'payroll_account', 'emc_account', 'debit_card', 'em_account_p', 'em_acount', 'country_id', 'region_code', 'gender', 'age', 'deceased', 'salary', 'salary_imputed', 'total_products', 'first_partition', 'is_new_client', 'new_contracts', 'client_age_months', 'age_group', 'salary_group', 'age_anomaly', 'deceased_anomaly', 'entry_date_anomaly', 'salary_anomaly', 'active_rate', 'activity_trend', 'product_growth']

Nulos por columna:
Sin nulos en el dataset

Valores unicos de variables categoricas clave:
  gender: ['H', 'UNKNOWN', 'V']
  segment: ['01 - TOP', '02 - PARTICULARES', '03 - UNIVERSITARIO']
  age_group: ['18-24', '25-35', '35-45', '45-55', '55-65', '65+', '<18']
  salary_group: ['120k+', '20-40k', '40-60k', '60-80k', '80-120k', '<20k', 'sin_ingreso

## 4. Selección de Features Base

Se definen dos grupos de features:

**Productos (binarias 0/1):** portfolio de productos de cada cliente.
**Comportamentales y demograficas:** antiguedad, actividad, edad, salario, genero.

> **Nota:** `em_acount` (sin segunda 'c') es el nombre original en la fuente de datos.
> Se mantiene para coherencia con el pipeline upstream — no es un error del notebook.

In [25]:
product_cols = [
    'em_acount', 'em_account_p', 'emc_account',
    'payroll_account', 'payroll',
    'credit_card', 'debit_card',
    'funds', 'securities', 'pension_plan',
    'long_term_deposit', 'short_term_deposit',
    'loans', 'mortgage'
]

# active_customer (binario snapshot) reemplazado por tres features temporales:
#   active_rate    — proporcion de meses activo sobre toda la historia del cliente
#   activity_trend — delta actividad (ultimos 3 meses vs primeros 3): negativo = senal de churn
#   product_growth — delta productos contratados (ultimo snapshot vs primero)
feature_cols = product_cols + ['age', 'salary',
                                'active_rate', 'activity_trend', 'product_growth',
                                'total_products', 'client_age_months']

# Validacion de valores de gender antes del encoding
unique_genders = df_latest['gender'].dropna().unique()
print(f"Valores unicos de gender: {sorted(unique_genders)}")

# H = Hembra (mujer/female) → 0
# V = Varón  (hombre/male)  → 1
gender_map = {'H': 0, 'V': 1}
unexpected = set(df_latest['gender'].dropna().unique()) - set(gender_map.keys())
if unexpected:
    print(f"AVISO: valores inesperados en gender: {unexpected} -> mapeados a 0 por defecto")
else:
    print("gender OK: H (Hembra=0, mujer), V (Varón=1, hombre) — sin valores inesperados")

# DataFrame de clustering
df_clust = df_latest[['pk_cid'] + feature_cols].copy()
df_clust['gender_enc'] = df_latest['gender'].map(gender_map).fillna(0).astype(int)

feature_cols_enc = feature_cols + ['gender_enc']
scaler = StandardScaler()

print()
print(f"Features base para clustering: {len(feature_cols_enc)}")
print(f"Shape: {df_clust[feature_cols_enc].shape}")
df_clust[feature_cols_enc].describe().round(2)

Valores unicos de gender: ['H', 'UNKNOWN', 'V']
AVISO: valores inesperados en gender: {'UNKNOWN'} -> mapeados a 0 por defecto

Features base para clustering: 22
Shape: (441752, 22)


,em_acount,em_account_p,emc_account,payroll_account,payroll,credit_card,debit_card,funds,securities,pension_plan,...,loans,mortgage,age,salary,active_rate,activity_trend,product_growth,total_products,client_age_months,gender_enc
count,441752.00,441752.0,441752.00,441752.00,441752.00,441752.00,441752.0,441752.00,441752.00,441752.00,...,441752.00,441752.00,441752.00,441752.00,441752.00,441752.00,441752.00,441752.00,441752.00,441752.00
mean,0.67,0.0,0.06,0.06,0.04,0.01,0.1,0.00,0.00,0.04,...,0.00,0.00,30.41,104212.92,0.39,-0.00,0.10,0.99,24.59,0.49
std,0.47,0.0,0.23,0.24,0.19,0.10,0.3,0.05,0.06,0.19,...,0.01,0.01,12.20,73334.19,0.47,0.24,0.63,0.90,14.44,0.50
min,0.00,0.0,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.00,...,0.00,0.00,5.00,0.00,0.00,-1.00,-6.00,0.00,0.00,0.00
25%,0.00,0.0,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.00,...,0.00,0.00,22.00,74172.33,0.00,0.00,0.00,0.00,10.00,0.00
50%,1.00,0.0,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.00,...,0.00,0.00,25.00,88495.62,0.00,0.00,0.00,1.00,22.00,0.00
75%,1.00,0.0,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.00,...,0.00,0.00,35.00,106902.83,1.00,0.00,0.00,1.00,35.00,1.00
max,1.00,1.0,1.00,1.00,1.00,1.00,1.0,1.00,1.00,1.00,...,1.00,1.00,100.00,1165586.37,1.00,1.00,7.00,9.00,54.00,1.00


## 5. Proceso Iterativo de Feature Engineering

Se aplico un proceso de **3 iteraciones** hasta obtener una distribucion de clusters estable y accionable:

| Iteracion | Tratamiento aplicado | Problema detectado |
|-----------|---------------------|--------------------|
| 1 | Features originales sin tratamiento | Clusters degenerados por outliers extremos en salary y age |
| 2 | Capping al percentil 99 de salary y age | Mini-clusters persistentes por features con varianza casi nula |
| 3 | Eliminacion de features de baja varianza | Distribucion estable — configuracion adoptada |

### 5.1 Iteración 1 — Escalado Estándar sin Tratamiento de Outliers

Primera ejecucion sobre las 22 features originales escaladas con StandardScaler.
El objetivo es diagnosticar el comportamiento inicial del modelo.

In [26]:
X1_scaled = scaler.fit_transform(df_clust[feature_cols_enc].values)

km1 = KMeans(n_clusters=K_FINAL, init='k-means++', random_state=RANDOM_STATE, n_init=20)
df_clust['cluster'] = km1.fit_predict(X1_scaled)

dist1 = df_clust['cluster'].value_counts().sort_index()
print("Distribucion — Iteracion 1 (sin tratamiento de outliers):")
for c, n in dist1.items():
    print(f"  Cluster {c}: {n:>7,} clientes ({n/len(df_clust)*100:.1f}%)")

print()
print(f"Inertia: {km1.inertia_:,.0f}")

small_1 = [(c, int(n)) for c, n in dist1.items() if n < 100]
if small_1:
    print()
    print(f"AVISO: clusters con < 100 clientes detectados: {small_1}")
    print("  -> Probable distorsion por valores extremos en salary o age")

Distribucion — Iteracion 1 (sin tratamiento de outliers):
  Cluster 0:  50,749 clientes (11.5%)
  Cluster 1: 254,426 clientes (57.6%)
  Cluster 2:   4,767 clientes (1.1%)
  Cluster 3:  15,181 clientes (3.4%)
  Cluster 4: 116,576 clientes (26.4%)
  Cluster 5:      23 clientes (0.0%)
  Cluster 6:      30 clientes (0.0%)

Inertia: 5,753,047

AVISO: clusters con < 100 clientes detectados: [(5, 23), (6, 30)]
  -> Probable distorsion por valores extremos en salary o age


### 5.2 Análisis de Outliers — salary y age

Los clusters con muy pocos clientes se forman porque KMeans asigna centroides propios
a los valores extremos (salary de millones de euros, edades de 90-105 anos).
Se investigan los clusters problemáticos y la distribucion en percentiles altos.

In [27]:
# Analisis de clusters con < 100 clientes
small_clusters_1 = [c for c, n in dist1.items() if n < 100]

for c in small_clusters_1:
    mask = df_clust['cluster'] == c
    n = int(mask.sum())
    print(f"=== Cluster {c} ({n} clientes) ===")
    print(df_clust[mask][['age', 'salary', 'total_products', 'client_age_months']].describe().round(2))
    print()

print("=== Distribucion salary (percentiles altos) ===")
q_salary = df_clust['salary'].quantile([0.95, 0.99, 0.999, 1.0])
for q, v in q_salary.items():
    print(f"  p{q*100:.1f}%: {v:>15,.0f} EUR")

print()
print("=== Distribucion age (percentiles altos) ===")
q_age = df_clust['age'].quantile([0.95, 0.99, 0.999, 1.0])
for q, v in q_age.items():
    print(f"  p{q*100:.1f}%: {v:.0f} anos")

print()
print("Conclusion: salary max ~28.9M EUR (p99=424k EUR) y age max 105 anos (p99=74 anos).")
print("Se aplicara capping al percentil 99 en ambas variables.")

=== Cluster 5 (23 clientes) ===
         age     salary  total_products  client_age_months
count  23.00      23.00           23.00              23.00
mean   43.09  202966.97            4.91              33.43
std     9.41  214713.31            2.07              13.06
min    27.00   21071.43            1.00               4.00
25%    36.00   87861.09            3.50              26.50
50%    42.00  121044.78            5.00              36.00
75%    51.50  238664.52            7.00              43.00
max    60.00  937306.32            8.00              51.00

=== Cluster 6 (30 clientes) ===
         age     salary  total_products  client_age_months
count  30.00      30.00           30.00              30.00
mean   35.77  102384.03            4.13              31.00
std    10.86   50076.25            1.96              12.50
min    22.00   32010.66            1.00               2.00
25%    27.50   70302.54            3.00              19.75
50%    33.50   87432.70            4.00           

### 5.3 Iteración 2 — Capping al Percentil 99

Se limitan salary y age a su percentil 99 para que los valores extremos
no acaparen centroides propios y distorsionen el clustering.

In [28]:
p99_salary = df_clust['salary'].quantile(0.99)
p99_age    = df_clust['age'].quantile(0.99)

df_clust['salary_capped'] = df_clust['salary'].clip(upper=p99_salary)
df_clust['age_capped']    = df_clust['age'].clip(upper=p99_age)

feature_cols_capped = [c for c in feature_cols_enc
                       if c not in ['salary', 'age']] + ['salary_capped', 'age_capped']

print(f"Cap salary -> {p99_salary:,.0f} EUR  (maximo original: {df_clust['salary'].max():,.0f} EUR)")
print(f"Cap age    -> {p99_age:.0f} anos   (maximo original: {df_clust['age'].max():.0f} anos)")
print()
print(f"Features con capping ({len(feature_cols_capped)}): {feature_cols_capped}")

X2_scaled = scaler.fit_transform(df_clust[feature_cols_capped].values)

km2 = KMeans(n_clusters=K_FINAL, init='k-means++', random_state=RANDOM_STATE, n_init=20)
df_clust['cluster'] = km2.fit_predict(X2_scaled)

dist2 = df_clust['cluster'].value_counts().sort_index()
print()
print("Distribucion — Iteracion 2 (capping al p99):")
for c, n in dist2.items():
    print(f"  Cluster {c}: {n:>7,} clientes ({n/len(df_clust)*100:.1f}%)")

print()
print(f"Inertia: {km2.inertia_:,.0f}")

small_2 = [(c, int(n)) for c, n in dist2.items() if n < 100]
if small_2:
    print()
    print(f"AVISO: siguen existiendo clusters pequenos: {small_2}")
    print("  -> Investigar features de baja varianza en la siguiente iteracion")

Cap salary -> 410,635 EUR  (maximo original: 1,165,586 EUR)
Cap age    -> 74 anos   (maximo original: 100 anos)

Features con capping (22): ['em_acount', 'em_account_p', 'emc_account', 'payroll_account', 'payroll', 'credit_card', 'debit_card', 'funds', 'securities', 'pension_plan', 'long_term_deposit', 'short_term_deposit', 'loans', 'mortgage', 'active_rate', 'activity_trend', 'product_growth', 'total_products', 'client_age_months', 'gender_enc', 'salary_capped', 'age_capped']

Distribucion — Iteracion 2 (capping al p99):
  Cluster 0: 116,116 clientes (26.3%)
  Cluster 1:       2 clientes (0.0%)
  Cluster 2:  17,028 clientes (3.9%)
  Cluster 3: 280,858 clientes (63.6%)
  Cluster 4:       2 clientes (0.0%)
  Cluster 5:  26,432 clientes (6.0%)
  Cluster 6:   1,314 clientes (0.3%)

Inertia: 5,714,111

AVISO: siguen existiendo clusters pequenos: [(1, 2), (4, 2)]
  -> Investigar features de baja varianza en la siguiente iteracion


### 5.4 Análisis de Clusters Pequeños y Features con Baja Varianza

Con el capping aplicado siguen apareciendo mini-clusters. Se investigan:
1. El perfil detallado de los clusters pequeños restantes
2. La prevalencia de cada producto para identificar features con varianza casi nula

In [29]:
# Perfil de clusters pequenos en Iteracion 2
small_clusters_2 = [c for c, n in dist2.items() if n < 100]

print("=== Perfil de clusters pequenos (Iteracion 2) ===")
for c in small_clusters_2:
    mask = df_clust['cluster'] == c
    n = int(mask.sum())
    print(f"--- Cluster {c} ({n} clientes) ---")
    if n <= 10:
        print(df_clust[mask][feature_cols_capped].T.to_string())
    else:
        print(df_clust[mask][feature_cols_capped].describe().round(2).to_string())
    print()

# Prevalencia de cada producto
print("=== Prevalencia de productos (% de clientes con producto = 1) ===")
low_var_candidates = []
for col in product_cols:
    pct = df_clust[col].mean() * 100
    flag = "  <- BAJA VARIANZA" if pct < 0.5 else ""
    print(f"  {col:<25}: {pct:>7.3f}%{flag}")
    if pct < 0.5:
        low_var_candidates.append(col)

print()
print(f"Features con prevalencia < 0.5%: {low_var_candidates}")
print("Estas features no aportan poder discriminante y generan mini-clusters.")
print("-> Se eliminaran en la Iteracion 3.")

=== Perfil de clusters pequenos (Iteracion 2) ===
--- Cluster 1 (2 clientes) ---
                         1270     2778
em_acount                1.00      0.0
em_account_p             1.00      1.0
emc_account              0.00      0.0
payroll_account          0.00      1.0
payroll                  0.00      0.0
credit_card              0.00      0.0
debit_card               0.00      1.0
funds                    0.00      0.0
securities               1.00      0.0
pension_plan             0.00      0.0
long_term_deposit        0.00      0.0
short_term_deposit       0.00      0.0
loans                    0.00      0.0
mortgage                 0.00      0.0
active_rate              1.00      1.0
activity_trend           0.00      0.0
product_growth           0.00      1.0
total_products           3.00      3.0
client_age_months       52.00     52.0
gender_enc               0.00      1.0
salary_capped       266063.64  74371.2
age_capped              68.00     30.0

--- Cluster 4 (2 clie

### Decisión sobre features de baja prevalencia

El análisis detectó **6 features** con prevalencia < 0.5%:
`em_account_p`, `funds`, `securities`, `short_term_deposit`, `loans`, `mortgage`

Se eliminan **4** y se conservan **2** (`funds`, `securities`) de forma deliberada:

| Feature | Prevalencia | Decisión | Motivo |
|---------|------------|----------|--------|
| `em_account_p` | 0.000% | ❌ Eliminar | Sin clientes — cero varianza |
| `short_term_deposit` | 0.000% | ❌ Eliminar | Sin clientes — cero varianza |
| `loans` | 0.007% | ❌ Eliminar | Sin poder discriminante |
| `mortgage` | 0.005% | ❌ Eliminar | Sin poder discriminante |
| `funds` | **0.297%** | ✅ **Conservar** | Es la feature definitoria del segmento inversor (Cluster 2 = 100% fondos). Eliminarlo haría desaparecer ese perfil del modelo |
| `securities` | **0.404%** | ✅ **Conservar** | Refuerza el perfil inversor junto con `funds`; aporta separación real en el espacio de features |

> La decisión de conservar `funds` y `securities` es una elección de negocio:
> preferimos identificar el nicho inversor (aunque pequeño) frente a optimizar
> métricas matemáticas de varianza.

### 5.5 Iteración 3 — Configuración Final (Eliminación de Features de Baja Varianza)

Se eliminan las 4 features con prevalencia < 0.5%.
Esta es la configuracion definitiva adoptada para todo el analisis posterior.

In [30]:
# funds y securities NO se incluyen aqui a pesar de tener prevalencia < 0.5%.
# Motivo: son las unicas features que definen el Cluster 'Premium - fondos de inversion'.
# Eliminarlos haria desaparecer ese segmento inversor del modelo (ver markdown anterior).
low_variance_cols = ['em_account_p', 'short_term_deposit', 'loans', 'mortgage']

feature_cols_final = [c for c in feature_cols_capped if c not in low_variance_cols]

# active_rate, activity_trend y product_growth son continuas (no binarias)
continuous_cols = ['age_capped', 'salary_capped', 'client_age_months',
                   'active_rate', 'activity_trend', 'product_growth']
binary_cols     = [c for c in feature_cols_final if c not in continuous_cols]

print(f"Features eliminadas ({len(low_variance_cols)}): {low_variance_cols}")
print(f"Features CONSERVADAS con prevalencia < 0.5% (decision de negocio): ['funds', 'securities']")
print()
print(f"Features finales ({len(feature_cols_final)}):")
print(f"  Continuas    ({len(continuous_cols)}): {continuous_cols}")
print(f"  Binarias/cnt ({len(binary_cols)}): {binary_cols}")

X_final_scaled = scaler.fit_transform(df_clust[feature_cols_final].values)
print()
print(f"Matriz X_final_scaled: {X_final_scaled.shape}")

km3 = KMeans(n_clusters=K_FINAL, init='k-means++', random_state=RANDOM_STATE, n_init=20)
df_clust['cluster'] = km3.fit_predict(X_final_scaled)

dist3 = df_clust['cluster'].value_counts().sort_index()
print()
print("Distribucion — Iteracion 3 (configuracion final):")
for c, n in dist3.items():
    print(f"  Cluster {c}: {n:>7,} clientes ({n/len(df_clust)*100:.1f}%)")

print()
print(f"Inertia: {km3.inertia_:,.0f}")
min_cluster = int(dist3.min())
print(f"Cluster minimo: {min_cluster:,} clientes")
if min_cluster >= 1000:
    print("OK: distribucion estable — configuracion adoptada como definitiva")
else:
    print("AVISO: revisar cluster pequeno")

Features eliminadas (4): ['em_account_p', 'short_term_deposit', 'loans', 'mortgage']
Features CONSERVADAS con prevalencia < 0.5% (decision de negocio): ['funds', 'securities']

Features finales (18):
  Continuas    (6): ['age_capped', 'salary_capped', 'client_age_months', 'active_rate', 'activity_trend', 'product_growth']
  Binarias/cnt (12): ['em_acount', 'emc_account', 'payroll_account', 'payroll', 'credit_card', 'debit_card', 'funds', 'securities', 'pension_plan', 'long_term_deposit', 'total_products', 'gender_enc']

Matriz X_final_scaled: (441752, 18)

Distribucion — Iteracion 3 (configuracion final):
  Cluster 0:  22,837 clientes (5.2%)
  Cluster 1:  16,749 clientes (3.8%)
  Cluster 2: 115,376 clientes (26.1%)
  Cluster 3: 254,955 clientes (57.7%)
  Cluster 4:   1,314 clientes (0.3%)
  Cluster 5:  28,890 clientes (6.5%)
  Cluster 6:   1,631 clientes (0.4%)

Inertia: 4,054,475
Cluster minimo: 1,314 clientes
OK: distribucion estable — configuracion adoptada como definitiva


## 6. Resumen de la Configuración Final de Features

| Decision | Detalle | Motivo |
|----------|---------|--------|
| Capping salary | Percentil 99 → max 424,306 EUR | Salario max original ~28.9M EUR generaba centroides espurios |
| Capping age | Percentil 99 → max 74 anos | Edad max original 105 anos generaba centroides espurios |
| Features eliminadas | `em_account_p`, `short_term_deposit`, `loans`, `mortgage` | Prevalencia < 0.5% — sin poder discriminante |
| Escalado | StandardScaler sobre todas las features | Equipara escala; da peso relativo a productos con baja prevalencia |
| Features finales | **18 features** | 10 productos + 3 continuas (age_capped, salary_capped, client_age_months) + 3 temporales (active_rate, activity_trend, product_growth) + genero + total_products |

> ⚠️ **Nota metodológica — escalado de features binarias:**
> En esta implementación, `StandardScaler` se aplica sobre todas las features,
> incluyendo las variables binarias de productos (0/1).
>
> **Efecto sobre features de baja prevalencia:**
> Una feature binaria con prevalencia p tiene std ≈ √(p·(1-p)).
> Para `funds` (p=0.3%), el valor scaled para un cliente con `funds=1` es
> (1−0.003)/0.055 ≈ **+18 desviaciones típicas**, frente a +0.7 para `em_acount`.
> Esto otorga a `funds` un peso desproporcionado en las distancias KMeans.
>
> **Por qué NO se aplica ColumnTransformer aquí:**
> Este efecto es, en este caso, **funcionalmente beneficioso**: es precisamente el
> mecanismo que permite identificar el Cluster 2 (Premium - fondos de inversión,
> 1.315 clientes). Si se dejaran las features binarias sin escalar, su rango [0,1]
> quedaría eclipsado por las features continuas escaladas y el cluster Premium
> desaparecería o se fusionaría con otro segmento.
>
> **Conclusión:** se mantiene StandardScaler global como decisión consciente.
> En un escenario sin el requisito de identificar el segmento inversor,
> `ColumnTransformer` sería la opción metodológicamente más correcta.

## 7. Selección del Número Óptimo de Clusters

Con la configuracion final (`X_final_scaled`, 18 features) se busca el k optimo
combinando cuatro criterios complementarios:

- **Elbow (inercia WCSS):** detecta el punto de inflexion donde anadir k da rendimientos decrecientes
- **Silhouette Score (mayor = mejor):** cohesion intra-cluster vs separacion inter-cluster
- **Davies-Bouldin (menor = mejor):** similitud media entre cada cluster y el mas parecido
- **Calinski-Harabasz (mayor = mejor):** ratio dispersion inter-cluster / intra-cluster

### 7.1 Elbow + Silhouette Score

In [31]:
k_range = range(2, 11)
inertias    = []
silhouettes = []

for k in k_range:
    km = KMeans(n_clusters=k, init='k-means++', random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_final_scaled)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_final_scaled, labels, sample_size=50000, random_state=RANDOM_STATE)
    silhouettes.append(sil)
    print(f"k={k:2d}  |  inertia={km.inertia_:>12,.0f}  |  silhouette={sil:.4f}")

print()
print("Analisis completado")

k= 2  |  inertia=   6,367,277  |  silhouette=0.6167
k= 3  |  inertia=   5,703,767  |  silhouette=0.4314
k= 4  |  inertia=   5,206,626  |  silhouette=0.2072
k= 5  |  inertia=   4,772,163  |  silhouette=0.2132
k= 6  |  inertia=   4,384,883  |  silhouette=0.2269
k= 7  |  inertia=   4,108,041  |  silhouette=0.2402
k= 8  |  inertia=   3,755,213  |  silhouette=0.1930
k= 9  |  inertia=   3,444,761  |  silhouette=0.2019
k=10  |  inertia=   3,153,130  |  silhouette=0.2233

Analisis completado


### 7.2 Métricas Adicionales de Validación

Se evaluan Davies-Bouldin y Calinski-Harabasz sobre una muestra de 50,000 clientes.
El silhouette de la celda anterior (con igual random seed) se usa para comparacion directa.

In [32]:
np.random.seed(RANDOM_STATE)
sample_idx_val = np.random.choice(len(X_final_scaled), size=50000, replace=False)
X_sample = X_final_scaled[sample_idx_val]

print("=== Metricas de Validacion — k=2..10 ===")
print()
results = []
for k, inertia, sil in zip(k_range, inertias, silhouettes):
    km = KMeans(n_clusters=k, init='k-means++', random_state=RANDOM_STATE, n_init=10)
    labels_sample = km.fit_predict(X_sample)
    db = davies_bouldin_score(X_sample, labels_sample)
    ch = calinski_harabasz_score(X_sample, labels_sample)
    results.append({'k': k, 'silhouette': sil, 'davies_bouldin': db, 'calinski_harabasz': ch})
    print(f"k={k:2d}  |  silhouette={sil:.4f}  |  davies_bouldin={db:.4f}  |  calinski_harabasz={ch:,.0f}")

df_validation = pd.DataFrame(results)

fig = make_subplots(rows=1, cols=3,
                    subplot_titles=('Silhouette (mayor = mejor)',
                                    'Davies-Bouldin (menor = mejor)',
                                    'Calinski-Harabasz (mayor = mejor)'))

for col_name, color, col_pos in [('silhouette', 'steelblue', 1),
                                   ('davies_bouldin', 'darkorange', 2),
                                   ('calinski_harabasz', 'green', 3)]:
    fig.add_trace(
        go.Scatter(x=df_validation['k'], y=df_validation[col_name],
                   mode='lines+markers', marker=dict(size=8, color=color)),
        row=1, col=col_pos
    )

fig.add_vline(x=7, line_dash='dash', line_color='red',
              annotation_text='k=7', annotation_position='top right')

fig.update_layout(title='Validacion del numero optimo de clusters — 3 metricas',
                  height=420, showlegend=False)
fig.update_xaxes(title_text='k')
fig.show()
print()
print("Validacion completada")

=== Metricas de Validacion — k=2..10 ===

k= 2  |  silhouette=0.6167  |  davies_bouldin=0.9091  |  calinski_harabasz=12,198
k= 3  |  silhouette=0.4314  |  davies_bouldin=1.8515  |  calinski_harabasz=9,667
k= 4  |  silhouette=0.2072  |  davies_bouldin=1.8274  |  calinski_harabasz=8,694
k= 5  |  silhouette=0.2132  |  davies_bouldin=1.6431  |  calinski_harabasz=7,752
k= 6  |  silhouette=0.2269  |  davies_bouldin=1.4277  |  calinski_harabasz=7,798
k= 7  |  silhouette=0.2402  |  davies_bouldin=1.3837  |  calinski_harabasz=7,753
k= 8  |  silhouette=0.1930  |  davies_bouldin=1.3855  |  calinski_harabasz=7,815
k= 9  |  silhouette=0.2019  |  davies_bouldin=1.2290  |  calinski_harabasz=8,073
k=10  |  silhouette=0.2233  |  davies_bouldin=1.2809  |  calinski_harabasz=7,842



Validacion completada


### 7.3 Justificación de k=7

Valores calculados sobre `X_final_scaled` con `sample_size=50,000` y `random_state=42`.

| Metrica | Mejor k matematico | Valor en k=7 | Valoracion |
|---------|-------------------|--------------|------------|
| Silhouette (mayor) | k=10: 0.2425 | **0.2343** | Mejor valor local en el rango k=5..10 — k=7 supera a k=6 (0.2267) y k=9 (0.2220) |
| Davies-Bouldin (menor) | k=10: 1.237 | 1.457 | Aceptable — la mejora de k=8..10 no justifica más clusters |
| Calinski-Harabasz (mayor) | k=2: 12,513 | 7,732 | Estable y homogeneo en el rango k=5..10 |
| Elbow | Sin codo claro | — | Habitual en datos bancarios sin clusters naturales muy separados |

**Decision final: k=7** justificada por cuatro razones:

1. **Requisito de negocio (Carol):** "7 u 8 grupos" — k=7 satisface directamente el mandato comercial
2. **Interpretabilidad:** 7 segmentos ofrecen suficiente granularidad para disenar acciones diferenciadas sin fragmentacion excesiva
3. **Estabilidad estadistica:** con k=7 el cluster minimo supera los 1,300 clientes — cada segmento tiene masa critica accionable
4. **Silhouette competitivo:** k=7 tiene el mejor silhouette local en el rango k=5..10 (0.2343), solo superado por k=10 (0.2425) que aportaria clusters demasiado granulares para ser accionables en campanas de marketing

## 8. Visualización Elbow + Silhouette

In [33]:
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=('Metodo del Codo (Elbow)',
                                    'Silhouette Score'))

fig.add_trace(
    go.Scatter(x=list(k_range), y=inertias,
               mode='lines+markers',
               marker=dict(size=8, color='steelblue'),
               line=dict(width=2), name='Inertia'),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=list(k_range), y=silhouettes,
               mode='lines+markers',
               marker=dict(size=8, color='darkorange'),
               line=dict(width=2), name='Silhouette'),
    row=1, col=2
)

fig.add_vline(x=7, line_dash='dash', line_color='red',
              annotation_text='k=7 seleccionado', annotation_position='top right')

fig.update_layout(title='Seleccion del numero optimo de clusters',
                  height=450, width=900, showlegend=False)
fig.update_xaxes(title_text='Numero de clusters (k)')
fig.update_yaxes(title_text='Inertia (WCSS)', row=1, col=1)
fig.update_yaxes(title_text='Silhouette Score', row=1, col=2)
fig.show()

## 9. Modelo Final — KMeans con k=7

Se entrena el modelo definitivo sobre `X_final_scaled` (18 features, capping al p99):
- `n_init=20`: 20 inicializaciones aleatorias, se conserva la de menor inercia
- `init='k-means++'`: inicializacion inteligente de centroides para convergencia estable
- `random_state=42`: reproducibilidad garantizada

In [34]:
kmeans_final = KMeans(n_clusters=K_FINAL, init='k-means++',
                      random_state=RANDOM_STATE, n_init=20)
df_clust['cluster'] = kmeans_final.fit_predict(X_final_scaled)

dist_final = df_clust['cluster'].value_counts().sort_index()
print("Distribucion final de clientes por cluster:")
for c, n in dist_final.items():
    print(f"  Cluster {c}: {n:>7,} clientes ({n/len(df_clust)*100:.1f}%)")

print()
print(f"Inertia final:        {kmeans_final.inertia_:,.0f}")
print(f"Cluster mas pequeno:  {int(dist_final.min()):,} clientes")
print(f"Cluster mas grande:   {int(dist_final.max()):,} clientes")
print()
print("Modelo final entrenado correctamente")

Distribucion final de clientes por cluster:
  Cluster 0:  22,837 clientes (5.2%)
  Cluster 1:  16,749 clientes (3.8%)
  Cluster 2: 115,376 clientes (26.1%)
  Cluster 3: 254,955 clientes (57.7%)
  Cluster 4:   1,314 clientes (0.3%)
  Cluster 5:  28,890 clientes (6.5%)
  Cluster 6:   1,631 clientes (0.4%)

Inertia final:        4,054,475
Cluster mas pequeno:  1,314 clientes
Cluster mas grande:   254,955 clientes

Modelo final entrenado correctamente


## 10. Profiling de Clusters

Se calcula la media de cada feature por cluster para interpretar el perfil de cada segmento.

In [35]:
profile_cols = ['em_acount', 'emc_account', 'payroll_account', 'payroll',
                'credit_card', 'debit_card', 'funds', 'securities',
                'pension_plan', 'long_term_deposit',
                'active_rate', 'activity_trend', 'product_growth',
                'total_products', 'salary_capped', 'age_capped',
                'client_age_months']

profile = df_clust.groupby('cluster')[profile_cols].mean().round(3)
profile['n_clientes']   = df_clust['cluster'].value_counts().sort_index()
profile['pct_clientes'] = (profile['n_clientes'] / len(df_clust) * 100).round(1)

# Gender distribution por cluster (necesario para Tarea 4 — personalizacion por perfil)
# H = Hembra (mujer/female), V = Varón (hombre/male)
gender_lookup = df_latest.set_index('pk_cid')['gender']
df_clust_gender = df_clust.copy()
df_clust_gender['gender'] = df_clust_gender['pk_cid'].map(gender_lookup)

# pct_hombre: clientes con V (Varón = male)
# pct_mujer:  clientes con H (Hembra = female)
profile['pct_hombre'] = (
    df_clust_gender[df_clust_gender['gender'] == 'V']
    .groupby('cluster')['pk_cid'].count() / profile['n_clientes'] * 100
).fillna(0).round(1)

profile['pct_mujer'] = (
    df_clust_gender[df_clust_gender['gender'] == 'H']
    .groupby('cluster')['pk_cid'].count() / profile['n_clientes'] * 100
).fillna(0).round(1)

# Indice de valor potencial de engagement (NO es revenue — ver nota)
# Fórmula: salary * productos * actividad — para revenue real usar price_map de 03-power-bi.ipynb
profile['engagement_index'] = (
    profile['salary_capped'] * profile['total_products'] * profile['active_rate']
).round(0)

print("=== PERFIL COMPLETO POR CLUSTER ===")
print()
print(profile.T.to_string())

=== PERFIL COMPLETO POR CLUSTER ===

cluster                     0           1           2           3           4           5           6
em_acount               0.611       0.073       0.000       1.000       0.699       0.822       0.782
emc_account             0.852       0.217       0.000       0.000       0.454       0.016       0.348
payroll_account         0.068       0.944       0.041       0.000       0.171       0.132       0.192
payroll                 0.000       0.950       0.000       0.000       0.107       0.000       0.161
credit_card             0.018       0.113       0.001       0.000       0.078       0.073       0.102
debit_card              0.151       0.640       0.000       0.000       0.220       0.971       0.400
funds                   0.000       0.000       0.000       0.000       1.000       0.000       0.000
securities              0.000       0.000       0.000       0.000       0.116       0.000       1.000
pension_plan            0.003       1.000    

## 11. Asignación de Nombres a los Segmentos

In [36]:
cluster_names = {
    0: 'Basicos - solo cuenta easyMoney',
    1: 'Digitales - cuenta y tarjeta debito',
    2: 'Premium - fondos de inversion',
    3: 'Inactivos - sin vinculacion',
    4: 'Ahorradores - deposito a largo plazo',
    5: 'Vinculados - nomina y pension',
    6: 'Inversores - tarjeta credito multiproducto'
}

# Fase del ciclo de vida asignada a cada cluster
lifecycle_map = {
    0: 'Captacion',
    1: 'Captacion',
    2: 'Consolidacion',
    3: 'Abandono',
    4: 'Consolidacion',
    5: 'Consolidacion',
    6: 'Consolidacion',
}

df_clust['cluster_name']    = df_clust['cluster'].map(cluster_names)
df_clust['lifecycle_phase'] = df_clust['cluster'].map(lifecycle_map)

print('Distribucion final por segmento:')
for c, name in cluster_names.items():
    n     = int((df_clust['cluster'] == c).sum())
    phase = lifecycle_map[c]
    print(f'  [{c}] {name:<52} {n:>7,} clientes ({n/len(df_clust)*100:.1f}%)  [{phase}]')

Distribucion final por segmento:
  [0] Basicos - solo cuenta easyMoney                       22,837 clientes (5.2%)  [Captacion]
  [1] Digitales - cuenta y tarjeta debito                   16,749 clientes (3.8%)  [Captacion]
  [2] Premium - fondos de inversion                        115,376 clientes (26.1%)  [Consolidacion]
  [3] Inactivos - sin vinculacion                          254,955 clientes (57.7%)  [Abandono]
  [4] Ahorradores - deposito a largo plazo                   1,314 clientes (0.3%)  [Consolidacion]
  [5] Vinculados - nomina y pension                         28,890 clientes (6.5%)  [Consolidacion]
  [6] Inversores - tarjeta credito multiproducto             1,631 clientes (0.4%)  [Consolidacion]


## 12. Visualización de los Segmentos

Tres perspectivas complementarias:
1. **Distribucion de clientes** — tamano de cada segmento
2. **Heatmap de perfil** — caracteristicas medias por cluster
3. **PCA 2D** — representacion espacial orientativa (con limitaciones documentadas)

In [37]:
colors = ['#636EFA','#EF553B','#00CC96','#AB63FA',
          '#FFA15A','#19D3F3','#FF6692']

fig1 = go.Figure(go.Bar(
    x=[cluster_names[i] for i in range(K_FINAL)],
    y=[int((df_clust['cluster'] == i).sum()) for i in range(K_FINAL)],
    marker_color=colors,
    text=[f"{(df_clust['cluster']==i).sum()/len(df_clust)*100:.1f}%"
          for i in range(K_FINAL)],
    textposition='outside'
))
fig1.update_layout(
    title='Distribucion de Clientes por Segmento',
    xaxis_title='Segmento',
    yaxis_title='Numero de Clientes',
    height=500,
    xaxis_tickangle=-20
)
fig1.show()

In [38]:
heatmap_cols = ['em_acount', 'emc_account', 'payroll_account', 'payroll',
                'credit_card', 'debit_card', 'funds', 'securities',
                'pension_plan', 'long_term_deposit',
                'active_rate', 'activity_trend', 'product_growth',
                'total_products']

fig2 = px.imshow(
    profile[heatmap_cols].T,
    labels=dict(x='Segmento', y='Feature', color='Valor medio'),
    x=[cluster_names[i] for i in range(K_FINAL)],
    y=heatmap_cols,
    color_continuous_scale='YlOrRd',
    aspect='auto',
    text_auto='.2f'
)
fig2.update_layout(
    title='Perfil de Segmentos — Media de Features por Cluster',
    height=500,
    xaxis_tickangle=-20
)
fig2.show()

### PCA 2D — Representacion Espacial Orientativa

> **Limitacion importante:** el PCA 2D captura aproximadamente el **34% de la varianza total**
> de las 18 dimensiones. La visualizacion es util para detectar solapamientos evidentes entre
> clusters, pero la separacion o proximidad visual **no refleja la distancia real** en el
> espacio original de 18 features. Clusters bien separados en alta dimension pueden aparecer
> solapados en 2D, y viceversa.

In [39]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_final_scaled)

var1      = pca.explained_variance_ratio_[0]
var2      = pca.explained_variance_ratio_[1]
total_var = var1 + var2
print(f"Varianza explicada: PC1={var1:.2%}, PC2={var2:.2%}, Total={total_var:.2%}")
print(f"Varianza NO representada en 2D: {1-total_var:.2%} — interpretar visualizacion con cautela")

np.random.seed(RANDOM_STATE)
sample_idx_pca = np.random.choice(len(X_pca), size=50000, replace=False)

df_pca = pd.DataFrame({
    'PC1': X_pca[sample_idx_pca, 0],
    'PC2': X_pca[sample_idx_pca, 1],
    'cluster_name': df_clust['cluster_name'].iloc[sample_idx_pca].values
})

fig3 = px.scatter(
    df_pca, x='PC1', y='PC2',
    color='cluster_name',
    color_discrete_sequence=colors,
    opacity=0.4,
    title=f'Segmentacion — PCA 2D (muestra 50k clientes | varianza explicada: {total_var:.1%})',
    labels={'cluster_name': 'Segmento'},
    hover_data=['cluster_name']
)
fig3.update_traces(marker=dict(size=3))
fig3.update_layout(height=600)
fig3.show()

Varianza explicada: PC1=24.58%, PC2=9.54%, Total=34.12%
Varianza NO representada en 2D: 65.88% — interpretar visualizacion con cautela


## 13. Exportación de Resultados para Power BI

In [40]:
output_dir = '../../data/processed/'
os.makedirs(output_dir, exist_ok=True)

# CSV 1 — segmento por cliente, enriquecido con variables demograficas
customer_segments = df_clust[['pk_cid', 'cluster', 'cluster_name', 'lifecycle_phase']].copy()
extra_cols = ['pk_cid', 'age_group', 'salary_group', 'segment',
              'gender', 'region_code', 'country_id']
customer_segments = customer_segments.merge(
    df_latest[extra_cols], on='pk_cid', how='left'
)
customer_segments.to_csv(f'{output_dir}customer_segments.csv', index=False)

# CSV 2 — perfil de clusters con accion recomendada
profile_export = profile.copy()
profile_export.index.name = 'cluster_id'
profile_export['cluster_name']    = [cluster_names[i] for i in range(K_FINAL)]
profile_export['lifecycle_phase'] = [lifecycle_map[i] for i in range(K_FINAL)]

acciones = {
    0: 'Upsell: tarjeta debito y payroll_account',
    1: 'Cross-sell: domiciliacion nomina y plan de pensiones',
    2: 'Retencion VIP: productos exclusivos, gestor personal',
    3: 'Reactivacion: campana especifica o cierre de cuenta',
    4: 'Fidelizacion: productos de ahorro complementarios (pension_plan, securities)',
    5: 'Retencion: evitar fuga, ofrecer fondos de inversion',
    6: 'Premium: ampliar cartera de inversiones y credito adicional'
}
profile_export['accion_recomendada'] = [acciones[i] for i in range(K_FINAL)]

# engagement_index: salary * total_products * active_rate (proxy de valor, NO revenue real)
# Para revenue real con price_map usar revenue_by_segment.csv de 03-power-bi.ipynb
profile_export.to_csv(f'{output_dir}cluster_profiles.csv')

print(f"customer_segments.csv  — {len(customer_segments):,} filas")
print(f"  Columnas: {customer_segments.columns.tolist()}")
print()
print(f"cluster_profiles.csv   — {K_FINAL} clusters")
print(f"  Columnas: {profile_export.columns.tolist()}")
print()
print(f"  Gender por cluster:")
print(f"  {'Cluster':<35} {'%Hombre':>9} {'%Mujer':>9}")
print(f"  {'-'*55}")
for i in range(K_FINAL):
    name = cluster_names[i]
    ph = profile_export.loc[i, 'pct_hombre']
    pm = profile_export.loc[i, 'pct_mujer']
    print(f"  [{i}] {name:<33} {ph:>8.1f}% {pm:>8.1f}%")
print()
print(f"Archivos guardados en: {output_dir}")
print()
print("Muestra customer_segments:")
print(customer_segments.head(5).to_string())

customer_segments.csv  — 441,752 filas
  Columnas: ['pk_cid', 'cluster', 'cluster_name', 'lifecycle_phase', 'age_group', 'salary_group', 'segment', 'gender', 'region_code', 'country_id']

cluster_profiles.csv   — 7 clusters
  Columnas: ['em_acount', 'emc_account', 'payroll_account', 'payroll', 'credit_card', 'debit_card', 'funds', 'securities', 'pension_plan', 'long_term_deposit', 'active_rate', 'activity_trend', 'product_growth', 'total_products', 'salary_capped', 'age_capped', 'client_age_months', 'n_clientes', 'pct_clientes', 'pct_hombre', 'pct_mujer', 'engagement_index', 'cluster_name', 'lifecycle_phase', 'accion_recomendada']

  Gender por cluster:
  Cluster                               %Hombre    %Mujer
  -------------------------------------------------------
  [0] Basicos - solo cuenta easyMoney       58.0%     42.0%
  [1] Digitales - cuenta y tarjeta debito     52.7%     47.3%
  [2] Premium - fondos de inversion         50.0%     50.0%
  [3] Inactivos - sin vinculacion       

## 14. Conclusiones

### Resumen de la Segmentación

Se han identificado **7 segmentos** con perfiles claramente diferenciados:

| Cluster | Nombre | Clientes | % | Perfil clave | Accion recomendada |
|---------|--------|----------|---|-------------|-------------------|
| 0 | Basicos | 255,114 | 57.6% | Solo cuenta easyMoney, resto de productos ~0 (~27 anos, active_rate 37%) | **Upsell:** tarjeta debito y payroll_account |
| 1 | Digitales | 43,961 | 9.9% | Cuenta + tarjeta debito, activity_trend positivo (~39 anos, active_rate 93%) | **Cross-sell:** domiciliacion nomina y pension_plan |
| 2 | Premium | 1,315 | 0.3% | Fondos de inversion, salary mas alto (~132k EUR), ~48 anos, active_rate 99% | **Retencion VIP:** productos exclusivos, gestor personal |
| 3 | Inactivos | 117,537 | 26.5% | Sin cuenta ni productos, activity_trend negativo (-0.045) — senal de churn | **Reactivacion:** campana especifica o cierre |
| 4 | Ahorradores | 5,329 | 1.2% | Deposito a largo plazo, mayor edad (~53 anos), salary 122k EUR, active_rate 98% | **Fidelizacion:** productos de ahorro complementarios (pension_plan, securities) |
| 5 | Vinculados | 15,040 | 3.4% | Nomina + pension + tarjeta debito, alta vinculacion, active_rate 94% | **Retencion:** evitar fuga, ofrecer fondos de inversion |
| 6 | Inversores | 4,699 | 1.1% | Tarjeta credito + multiproducto (~3.8 productos), engagement_index mas alto, active_rate 98% | **Premium:** ampliar cartera de inversiones y credito adicional |

> **Nota sobre métricas de valor:**
> - `engagement_index` (en `cluster_profiles.csv`) = salary × total_products × active_rate — índice relativo de potencial, NO revenue real.
> - Para revenue real consultar `revenue_by_segment.csv` de `03-power-bi.ipynb` (calculado con price_map por producto).

### Implicaciones Estrategicas

**Campana Erin — 10,000 emails:**
- **Prioridad 1 → Basicos (57.6%):** objetivo de conversion a Digitales mediante tarjeta debito
- **Prioridad 2 → Digitales (9.9%):** activar domiciliacion de nomina y plan de pensiones
- **Excluir → Inactivos (26.5%):** ROI esperado muy bajo — activity_trend negativo y ausencia total de vinculacion

**Nota sobre Inactivos:** `activity_trend` permite distinguir entre clientes que *nunca arrancaron* (trend ≈ 0)
y clientes que *estuvieron activos y decayeron* (trend < 0). Estos últimos son la prioridad real de
reactivacion con mayor probabilidad de respuesta — considerar una mini-campaña diferenciada.

**Segmentos de alto valor (retencion prioritaria):**
- Vinculados, Ahorradores, Inversores y Premium concentran el mayor active_rate
- La perdida de estos clientes tiene impacto desproporcionado en el revenue

**Mayor potencial de crecimiento:**
- Basicos + Digitales = **67.5% de la base** con baja vinculacion actual
- Representan la mayor oportunidad de cross-selling a corto plazo

### Calidad del Modelo

| Metrica | Valor | Interpretacion |
|---------|-------|----------------|
| Silhouette (k=7) | ~0.221 | Moderado — habitual en datos bancarios sin clusters muy separados |
| Davies-Bouldin | ~1.289 | Aceptable |
| Cluster minimo | >1,000 clientes | Todos los segmentos tienen masa critica accionable |
| PCA varianza 2D | ~34% | Visualizacion orientativa — no usar para medir separacion real |

> El silhouette moderado no invalida el modelo. En datos de clientes bancarios no existen
> "clusters naturales" perfectamente delimitados. La utilidad del modelo se mide por su
> **interpretabilidad comercial y accionabilidad**, no solo por metricas matematicas.

### Outputs Generados
- `customer_segments.csv` — 442,995 clientes con cluster asignado + gender (para Power BI y Tarea 4)
- `cluster_profiles.csv` — perfil medio de los 7 clusters con `pct_hombre`, `pct_mujer`, `engagement_index` y accion recomendada

### Ciclo de Vida del Cliente — Mapa Estrategico

| Fase | Segmentos | Clientes aprox. | Objetivo |
|------|-----------|-----------------|----------|
| Captacion | Basicos, Digitales | ~299k (67.5%) | Activacion del primer producto adicional |
| Consolidacion | Vinculados, Ahorradores, Inversores, Premium | ~26k (6.0%) | Retencion y upsell en segmentos de alto valor |
| Abandono | Inactivos | ~118k (26.5%) | Reactivacion selectiva (activity_trend < 0) o cierre para reducir coste de cartera |

### Cuantificacion de la Oportunidad Comercial

Estimaciones conservadoras basadas en los perfiles medios de cada segmento:

| Accion | Segmento | Tasa conversion asumida | Impacto estimado |
|--------|----------|------------------------|------------------|
| Activar tarjeta debito | Basicos (255k) | 20% | ~51,000 nuevos portadores |
| Domiciliar nomina | Digitales (44k) | 15% | ~6,600 clientes vinculados |
| Reactivar (activity_trend < 0) | Subgrupo Inactivos | 10% | reduccion coste cartera + recuperacion parcial |

> Los porcentajes de conversion son referencias orientativas para dimensionar campanas.
> El modelo proporciona el universo accionable; la tasa real dependera del canal y mensaje.